# GRAND sweep — DART (n x rho) + LRao, resumable checkpoints every 100 ep
Spec (08-13): full 16-rho sweep at n=2048 (and pavia4's fixed pool); rho {.005,.03,.1}
at other n. 30000 ep DART / 5000 ep LRao, NO best-epoch selection (sparse pd curve
recorded for plots only). fp16 weight snapshot every `CKPT_EVERY`; FULL resume state
(weights+optimizer+RNG+loss) every `FULL_STATE_EVERY` — rerunning any cell resumes
interrupted runs exactly. Run ONE setting per Colab session (SETTINGS knob).
Order: CONFIG -> DART engine -> LRao engine -> plots -> archive.

In [ ]:
!git clone -b rebuttal --depth 1 https://github.com/michaelpiro/final-paper-experiment.git repo
%cd repo
import os, torch
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')
assert os.path.exists('repro/data/pavia-u.mat'), 'missing data'

In [ ]:
# ======================= CONFIG — edit me =======================
SETTINGS = ['multi']            # one per session: 'multi' | 'single' | 'pavia4'
N_LIST   = [32, 64, 128, 256, 512, 1024, 2048]
RHOS_FULL  = [0.0001, 0.0005, 0.001, 0.003, 0.005, 0.007, 0.01, 0.03,
              0.05, 0.071, 0.1, 0.3, 0.5, 0.7, 1.0, 2.0]   # n=2048 + pavia4
RHOS_SMALL = [0.005, 0.03, 0.1]                             # all other n
SEEDS  = [42]
EPOCHS = 30000
CKPT_EVERY       = 100    # fp16 weight snapshot cadence
FULL_STATE_EVERY = 1000   # exact-resume state (weights+opt+RNG+loss)
EVAL_PD_EVERY    = 500    # sparse pd curve (plots only, NO selection)
FRONT  = 'std'
LR, WD, CLIP, BATCH = 5e-4, 0.0, 1.0, 512
SINGLE_LINEAR = True      # single also trains linear DART (L-DART arch)
PFA = 0.1
# ---- LRao (no val, no ES, full batch = the regularizer) ----
LRAO_EPOCHS     = 5000
LRAO_CKPT_EVERY = 100
LRAO_EVAL_EVERY = 100
LRAO_WD, LRAO_CLIP = 0.0, 0.0
# ---- paths ----
OUT = 'results_grand.json'
CKPT_ROOT = 'ckpt_grand'
PLOTS = 'plots'
# =================================================================

In [ ]:
# DART ENGINE — resumable; edit CONFIG above, run this cell (re-run to resume)
import copy, json, yaml
import numpy as np, torch
from tqdm import tqdm
from repro.protocols.iid import load_hsi, build_pools, _pd_at_fa, _auc
from repro.core.data import Whitening, plant_targets
from repro.core.models import ScoreNet
from repro import scenes
from repro.protocols.spatial import load_cfg as load_sp_cfg

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.set_grad_enabled(True)
os.makedirs(CKPT_ROOT, exist_ok=True)

_POOLS, _CFGS, _SP = {}, {}, {}
def iid_cfg(mode):
    if mode not in _CFGS:
        c = yaml.safe_load(open(f'repro/configs/iid_{mode}.yaml'))
        c['dataset'] = 'repro/data/pavia-u.mat'
        _CFGS[mode] = c
    return _CFGS[mode]

def pools(setting, seed, n):
    key = (setting, seed, n)
    if key in _POOLS: return _POOLS[key]
    if setting == 'pavia4':
        if 'sc' not in _SP:
            _SP['cfg'] = load_sp_cfg()
            _SP['sc'] = scenes.build('pavia4', _SP['cfg'])
        sc, spc = _SP['sc'], _SP['cfg']
        tr = sc['tr'].astype(np.float32)
        s = np.asarray(sc['sig'], np.float32)
        planted, labels, _ = plant_targets(
            sc['te'], s, 0.15, float(spc['target_fraction']), model='additive',
            seed=seed, spatial_shape=sc['te_shape'],
            edge_guard=int(spc['edge_guard']))
    else:
        c = iid_cfg(setting)
        data, gt = load_hsi(c['dataset'])
        bkg, tgt = build_pools(data, gt.flatten(), c, setting)
        s = tgt.mean(axis=0).astype(np.float32)
        rng = np.random.default_rng(seed)
        idx = np.arange(len(bkg)); rng.shuffle(idx)
        shuf = bkg[idx]
        tr = shuf[:n].astype(np.float32)
        te = shuf[-int(c['test_size']):].astype(np.float32)
        planted, labels, _ = plant_targets(te, s, float(c['amplitude']),
                                           c['target_fraction'],
                                           model='additive', seed=seed)
    _POOLS[key] = (tr, planted.astype(np.float32), np.asarray(labels), s)
    return _POOLS[key]

def make_front(tr):
    X = np.asarray(tr, np.float64)
    if FRONT == 'robust':
        med = np.median(X, axis=0)
        sc_ = np.median(np.abs(X - med), axis=0) * 1.4826
        return Whitening(med.astype(np.float32), np.diag(1.0/sc_).astype(np.float32))
    if FRONT == 'std':
        return Whitening(X.mean(0).astype(np.float32),
                         np.diag(1.0/X.std(0)).astype(np.float32))
    lam, V = np.linalg.eigh(np.cov(X, rowvar=False))
    return Whitening(X.mean(0).astype(np.float32),
                     (V @ np.diag(1.0/np.sqrt(lam)) @ V.T).astype(np.float32))

def rhos_for(n):
    return RHOS_FULL if (n == 2048 or n == 'scene') else RHOS_SMALL

def latest_full(rundir):
    p = os.path.join(rundir, 'resume.pt')
    return p if os.path.exists(p) else None

def run_dart(setting, arch, n, rho, seed):
    key = f'dart-{arch}_{setting}_n{n}_r{rho}_s{seed}'
    res = json.load(open(OUT)) if os.path.exists(OUT) else {}
    if res.get(key, {}).get('epochs_done', 0) >= EPOCHS:
        print('skip (done):', key); return
    tr, planted, y, s = pools(setting, seed, n)
    D = tr.shape[1]
    c = iid_cfg(setting if setting != 'pavia4' else 'multi')
    hidden = [] if arch == 'lin' else [128]
    act = c.get('activation', 'silu') if arch == 'lin' else 'relu'
    sigma = float(np.sqrt(rho * np.asarray(tr, np.float64).var(0).mean()))
    torch.manual_seed(seed)
    net = ScoreNet(D, hidden, act, whitening=make_front(tr)).to(DEVICE)
    opt = torch.optim.Adam(net.parameters(), lr=LR, weight_decay=WD)
    gen = torch.Generator(device=DEVICE); gen.manual_seed(97 * seed)
    rundir = os.path.join(CKPT_ROOT, key)
    os.makedirs(rundir, exist_ok=True)
    ep0, loss_hist, pd_hist = 0, [], []
    lf = latest_full(rundir)
    if lf:
        blob = torch.load(lf, map_location=DEVICE, weights_only=False)
        net.load_state_dict(blob['net']); opt.load_state_dict(blob['opt'])
        gen.set_state(blob['gen'].cpu())
        ep0 = blob['epoch']; loss_hist = [float(v) for v in blob['loss_hist']]
        pd_hist = list(blob['pd_hist'])
        print(f'  resume {key} @ {ep0}')
    X = torch.tensor(tr, device=DEVICE)
    Pt = torch.tensor(planted, device=DEVICE)
    nn_ = len(X)

    def psi(A):
        out = []
        with torch.no_grad():
            for i in range(0, len(A), 4096):
                out.append(net(A[i:i+4096]).cpu().numpy())
        return np.concatenate(out, 0)

    def save_full(ep):
        # rolling EXACT-resume state (weights + Adam moments + RNG + histories)
        torch.save({'net': net.state_dict(), 'opt': opt.state_dict(),
                    'gen': gen.get_state(), 'epoch': ep,
                    'loss_hist': np.asarray(loss_hist, np.float32),
                    'pd_hist': pd_hist, 'key': key, 'sigma': sigma,
                    'setting': setting, 'arch': arch, 'n': n, 'rho': rho,
                    'seed': seed, 'front': FRONT, 'lr': LR, 'wd': WD,
                    'clip': CLIP}, os.path.join(rundir, 'resume.pt'))

    bar = tqdm(range(ep0 + 1, EPOCHS + 1), desc=key, dynamic_ncols=True,
               mininterval=5.0, ascii=True, initial=ep0, total=EPOCHS)
    for ep in bar:
        net.train()
        perm = torch.randperm(nn_, generator=gen, device=DEVICE)
        tot, nb = 0.0, 0
        for i in range(0, nn_, BATCH):
            b = X[perm[i:i+BATCH]]
            eps = torch.randn(b.shape, generator=gen, device=DEVICE) * sigma
            loss = ((net(b + eps) + eps / sigma**2)**2).sum(-1).mean()
            opt.zero_grad(); loss.backward()
            if CLIP: torch.nn.utils.clip_grad_norm_(net.parameters(), CLIP)
            opt.step()
            tot += float(loss.detach()); nb += 1
        loss_hist.append(tot / max(nb, 1))
        if ep % EVAL_PD_EVERY == 0 or ep == EPOCHS:
            net.eval()
            z_tr, z_te = psi(X), psi(Pt)
            zb = z_tr.mean(0); Cz = np.cov(z_tr, rowvar=False)
            T = -((z_te - zb) @ s) / np.sqrt(float(s @ Cz @ s))
            pd_hist.append({'epoch': ep,
                            'pd': round(float(_pd_at_fa(y, T, PFA)), 4),
                            'auc': round(float(_auc(y, T)), 4)})
            bar.set_postfix_str(f'loss={loss_hist[-1]:.3g} '
                                f'pd={pd_hist[-1]["pd"]:.3f}')
        if ep % CKPT_EVERY == 0:
            # pick-and-continue point: weights (fp16) + epoch + RNG + loss so far
            torch.save({'w16': {k: v.half().cpu() for k, v in
                                net.state_dict().items()}, 'epoch': ep,
                        'gen': gen.get_state(),
                        'loss_tail': np.asarray(loss_hist[-CKPT_EVERY:],
                                                np.float32)},
                       os.path.join(rundir, f'snap_{ep:06d}.pt'))
        if ep % FULL_STATE_EVERY == 0 or ep == EPOCHS:
            save_full(ep)
    bar.close()
    res = json.load(open(OUT)) if os.path.exists(OUT) else {}
    res[key] = {'det': f'dart-{arch}', 'setting': setting, 'n': n, 'rho': rho,
                'seed': seed, 'front': FRONT, 'epochs_done': EPOCHS,
                'final_pd': pd_hist[-1]['pd'], 'final_auc': pd_hist[-1]['auc'],
                'pd_hist': pd_hist, 'loss100': [round(float(v), 5) for v in
                                                loss_hist[::100]]}
    json.dump(res, open(OUT, 'w'), indent=1)
    print(f'[{key}] final_pd={pd_hist[-1]["pd"]:.3f} '
          f'final_auc={pd_hist[-1]["auc"]:.3f}', flush=True)

for setting in SETTINGS:
    ns = ['scene'] if setting == 'pavia4' else N_LIST
    archs = ['mlp'] + (['lin'] if setting == 'single' and SINGLE_LINEAR else [])
    for n in ns:
        for rho in rhos_for(n):
            for seed in SEEDS:
                for arch in archs:
                    run_dart(setting, arch, n, rho, seed)
print('DART ENGINE DONE')

In [ ]:
# LRao ENGINE — no val, no ES, full batch; resumable; ckpt every LRAO_CKPT_EVERY
from repro.protocols.iid import _make_whitening
from repro.core.models import (compute_lfi_detector_scores_mode2,
                               lfi_loss_mode2, ScoreNet)
from repro.core.normalization import robust_whitening_iqr

def run_lrao(setting, n, seed):
    key = f'lrao_{setting}_n{n}_s{seed}'
    res = json.load(open(OUT)) if os.path.exists(OUT) else {}
    if res.get(key, {}).get('epochs_done', 0) >= LRAO_EPOCHS:
        print('skip (done):', key); return
    tr, planted, y, s = pools(setting, seed, n)
    c = iid_cfg(setting if setting != 'pavia4' else 'multi')
    lcfg = dict(c)
    lcfg['hidden_dims'] = [128]   # LRao-MLP, paper width (yaml field roles differ per mode)
    lcfg['activation'] = 'relu'
    norm = str(lcfg.get('lrao_input_norm', 'robust'))
    Wl = (robust_whitening_iqr(tr) if norm == 'robust' else
          _make_whitening(tr, {**lcfg, 'whiten_eig_floor':
              lcfg.get('lrao_whiten_eig_floor', lcfg.get('whiten_eig_floor', 0.0))}))
    torch.manual_seed(seed)
    model = ScoreNet(tr.shape[1], lcfg['hidden_dims'], lcfg['activation'],
                     whitening=Wl).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=float(lcfg['lr']),
                           weight_decay=LRAO_WD)
    rundir = os.path.join(CKPT_ROOT, key)
    os.makedirs(rundir, exist_ok=True)
    ep0, trj_hist, pd_hist = 0, [], []
    lf = latest_full(rundir)
    if lf:
        blob = torch.load(lf, map_location=DEVICE, weights_only=False)
        model.load_state_dict(blob['net']); opt.load_state_dict(blob['opt'])
        torch.set_rng_state(blob['torch_rng'].cpu())
        ep0 = blob['epoch']; trj_hist = [float(v) for v in blob['trj_hist']]
        pd_hist = list(blob['pd_hist'])
        print(f'  resume {key} @ {ep0}')
    Xl = torch.tensor(tr, device=DEVICE)
    nn_ = len(Xl)
    dth = float(lcfg['lfi_delta_theta'])
    bar = tqdm(range(ep0 + 1, LRAO_EPOCHS + 1), desc=key, dynamic_ncols=True,
               mininterval=5.0, ascii=True, initial=ep0, total=LRAO_EPOCHS)
    for ep in bar:
        model.train()
        try:
            loss = lfi_loss_mode2(model, Xl, dth,
                                  detach_sigma=lcfg['lfi_detach_sigma'])
            ok = torch.isfinite(loss)
        except Exception:
            ok = False
        if ok:
            opt.zero_grad(); loss.backward()
            if LRAO_CLIP: torch.nn.utils.clip_grad_norm_(model.parameters(),
                                                         LRAO_CLIP)
            opt.step()
            trj_hist.append(-float(loss.detach()))
        else:
            trj_hist.append(float('nan'))
        if ep % LRAO_EVAL_EVERY == 0 or ep == LRAO_EPOCHS:
            model.eval()
            T = np.asarray(compute_lfi_detector_scores_mode2(
                model, tr, planted, s, dth))
            pd_hist.append({'epoch': ep,
                            'pd': round(float(_pd_at_fa(y, T, PFA)), 4),
                            'auc': round(float(_auc(y, T)), 4)})
            bar.set_postfix_str(f'trJ={trj_hist[-1]:.4g} '
                                f'pd={pd_hist[-1]["pd"]:.3f}')
        if ep % LRAO_CKPT_EVERY == 0:
            torch.save({'w16': {k: v.half().cpu() for k, v in
                                model.state_dict().items()}, 'epoch': ep,
                        'torch_rng': torch.get_rng_state(),
                        'trj_tail': np.asarray(trj_hist[-LRAO_CKPT_EVERY:],
                                               np.float32)},
                       os.path.join(rundir, f'snap_{ep:06d}.pt'))
        if ep % 500 == 0 or ep == LRAO_EPOCHS:
            torch.save({'net': model.state_dict(), 'opt': opt.state_dict(),
                        'torch_rng': torch.get_rng_state(), 'epoch': ep,
                        'trj_hist': np.asarray(trj_hist, np.float32),
                        'pd_hist': pd_hist, 'key': key, 'norm': norm,
                        'setting': setting, 'n': n, 'seed': seed},
                       os.path.join(rundir, 'resume.pt'))
    bar.close()
    res = json.load(open(OUT)) if os.path.exists(OUT) else {}
    res[key] = {'det': 'lrao', 'setting': setting, 'n': n, 'rho': None,
                'seed': seed, 'epochs_done': LRAO_EPOCHS, 'norm': norm,
                'final_pd': pd_hist[-1]['pd'], 'final_auc': pd_hist[-1]['auc'],
                'pd_hist': pd_hist,
                'trj100': [round(float(v), 5) for v in trj_hist[::100]]}
    json.dump(res, open(OUT, 'w'), indent=1)
    print(f'[{key}] final_pd={pd_hist[-1]["pd"]:.3f}', flush=True)

for setting in SETTINGS:
    ns = ['scene'] if setting == 'pavia4' else N_LIST
    for n in ns:
        for seed in SEEDS:
            run_lrao(setting, n, seed)
print('LRAO ENGINE DONE')

In [ ]:
# PLOT 1 — IID n-sweep: DART envelope + fixed-rho lines + LRao + baselines
import json, numpy as np, matplotlib.pyplot as plt
from repro.core.detectors import amf, gmm_glrt_levin_additive
os.makedirs(PLOTS, exist_ok=True)
res = json.load(open(OUT))
RHO_LINES = [0.03, 0.01, 0.005]      # .01 exists only at n=2048
COL = {'env': '#2a78d6', 0.03: '#eb6834', 0.01: '#1baf7a', 0.005: '#eda100',
       'lrao': '#9467bd', 'AMF': '#666666', 'GMM-Levin': '#999999'}
for setting in [s for s in SETTINGS if s != 'pavia4']:
    fig, ax = plt.subplots(figsize=(8.5, 5.2), dpi=120)
    def cells(pred):
        out = {}
        for v in res.values():
            if v['setting'] == setting and pred(v):
                out.setdefault(v['n'], []).append(v['final_pd'])
        return out
    env = {}
    for v in res.values():
        if v['setting'] == setting and v['det'] == 'dart-mlp':
            env.setdefault(v['n'], []).append(v['final_pd'])
    ns = sorted(env)
    ax.plot(ns, [max(env[n]) for n in ns], color=COL['env'], lw=2.5,
            marker='o', label='DART envelope (best rho, final ep)')
    for r in RHO_LINES:
        c = cells(lambda v, r=r: v['det'] == 'dart-mlp' and v['rho'] == r)
        if c:
            ns2 = sorted(c)
            ax.plot(ns2, [np.mean(c[n]) for n in ns2], color=COL[r], lw=1.8,
                    marker='s', ms=4, label=f'DART rho={r} (final ep)')
    c = cells(lambda v: v['det'] == 'lrao')
    if c:
        ns2 = sorted(c)
        ax.plot(ns2, [np.mean(c[n]) for n in ns2], color=COL['lrao'], lw=2,
                marker='^', label='LRao (final ep)')
    cfg_ = iid_cfg(setting); theta = float(cfg_['amplitude'])
    for det, fn in (('AMF', amf), ('GMM-Levin', gmm_glrt_levin_additive)):
        if det == 'GMM-Levin' and setting == 'single': continue
        mu = []
        for n in N_LIST:
            vals = []
            for seed in SEEDS:
                tr, planted, y, s = pools(setting, seed, n)
                T = np.asarray(fn(planted, tr, s))
                v = float(_pd_at_fa(y, T, PFA))
                if np.isfinite(v): vals.append(v)
            mu.append(np.mean(vals) if vals else np.nan)
        ax.plot(N_LIST, mu, color=COL[det], lw=1.5, ls='--', label=det)
    ax.set_xscale('log'); ax.set_xlabel('n train')
    ax.set_ylabel(f'Pd @ Pfa={PFA} (final epoch)')
    ax.set_title(f'{setting} n-sweep — final-checkpoint models')
    ax.grid(alpha=.25); ax.legend(frameon=False, fontsize=8)
    plt.tight_layout(); plt.savefig(f'{PLOTS}/nsweep_{setting}.png', dpi=150)
    plt.show()

In [ ]:
# PLOT 2 — rho sweep at n=2048 (and pavia4 scene): final-epoch pd vs rho
res = json.load(open(OUT))
fig, ax = plt.subplots(figsize=(8.5, 5), dpi=120)
colors = {'multi': '#2a78d6', 'single': '#eb6834',
          'single-lin': '#1baf7a', 'pavia4': '#eda100'}
for setting in SETTINGS:
    for det, tag in (('dart-mlp', setting), ('dart-lin', f'{setting}-lin')):
        pts = {}
        for v in res.values():
            if (v['setting'] == setting and v['det'] == det and
                    v['n'] in (2048, 'scene')):
                pts.setdefault(v['rho'], []).append(v['final_pd'])
        if not pts: continue
        rs = sorted(pts)
        ax.plot(rs, [np.mean(pts[r]) for r in rs], marker='o', lw=2,
                color=colors.get(tag, '#333333'), label=f'{tag} ({det})')
ax.set_xscale('log'); ax.set_xlabel('rho'); ax.set_ylabel('final-epoch Pd')
ax.set_title('rho response @ n=2048 / pavia4 scene (30000 ep, final ckpt)')
ax.grid(alpha=.25); ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(f'{PLOTS}/rhosweep_2048.png', dpi=150)
plt.show()

In [ ]:
# PLOT 3 — loss curves (DART loss100 + LRao trJ) for n = 2048, 512, 128
res = json.load(open(OUT))
NSHOW = [2048, 512, 128]
for setting in [s for s in SETTINGS if s != 'pavia4']:
    fig, axes = plt.subplots(1, len(NSHOW), figsize=(14, 4), dpi=120)
    for ax, n in zip(axes, NSHOW):
        for v in res.values():
            if (v['setting'] == setting and v['det'] == 'dart-mlp'
                    and v['n'] == n and v['rho'] in (0.005, 0.03, 0.1)):
                xs = np.arange(len(v['loss100'])) * 100
                ax.plot(xs, v['loss100'], lw=1.5, label=f"rho={v['rho']}")
        ax.set_yscale('log'); ax.set_title(f'{setting} n={n} — DART loss')
        ax.set_xlabel('epoch'); ax.grid(alpha=.25)
        ax.legend(frameon=False, fontsize=7)
    plt.tight_layout(); plt.savefig(f'{PLOTS}/loss_dart_{setting}.png', dpi=150)
    plt.show()
    fig, axes = plt.subplots(1, len(NSHOW), figsize=(14, 4), dpi=120)
    for ax, n in zip(axes, NSHOW):
        for v in res.values():
            if v['setting'] == setting and v['det'] == 'lrao' and v['n'] == n:
                xs = np.arange(len(v['trj100'])) * 100
                ax.plot(xs, v['trj100'], lw=1.5, color='#9467bd')
        ax.set_title(f'{setting} n={n} — LRao tr(J*)')
        ax.set_xlabel('epoch'); ax.grid(alpha=.25)
    plt.tight_layout(); plt.savefig(f'{PLOTS}/loss_lrao_{setting}.png', dpi=150)
    plt.show()

In [ ]:
# PLOT 4 — pd vs epoch for n = 2048, 512, 128: DART (rho .005/.03/.1) + LRao
res = json.load(open(OUT))
NSHOW = [2048, 512, 128]
for setting in [s for s in SETTINGS if s != 'pavia4']:
    fig, axes = plt.subplots(1, len(NSHOW), figsize=(14, 4), dpi=120)
    for ax, n in zip(axes, NSHOW):
        for v in res.values():
            if v['setting'] != setting or v['n'] != n: continue
            if v['det'] == 'dart-mlp' and v['rho'] in (0.005, 0.03, 0.1):
                ax.plot([p['epoch'] for p in v['pd_hist']],
                        [p['pd'] for p in v['pd_hist']], lw=1.5,
                        label=f"DART rho={v['rho']}")
            elif v['det'] == 'lrao':
                ax.plot([p['epoch'] for p in v['pd_hist']],
                        [p['pd'] for p in v['pd_hist']], lw=2,
                        color='#9467bd', label='LRao')
        ax.set_title(f'{setting} n={n}'); ax.set_xlabel('epoch')
        ax.set_ylabel(f'Pd@{PFA}'); ax.grid(alpha=.25)
        ax.legend(frameon=False, fontsize=7)
    plt.tight_layout(); plt.savefig(f'{PLOTS}/pd_vs_epoch_{setting}.png', dpi=150)
    plt.show()

In [ ]:
# ARCHIVE — zip EVERYTHING (results + all checkpoints + plots).
# Mount Google Drive first for large archives:  from google.colab import drive; drive.mount('/content/drive')
import shutil, os
tag = '_'.join(SETTINGS)
zips = []
shutil.make_archive(f'grand_results_{tag}', 'zip', '.', OUT)
zips.append(f'grand_results_{tag}.zip')
if os.path.isdir(PLOTS) and os.listdir(PLOTS):
    shutil.make_archive(f'grand_plots_{tag}', 'zip', '.', PLOTS)
    zips.append(f'grand_plots_{tag}.zip')
if os.path.isdir(CKPT_ROOT):
    shutil.make_archive(f'grand_ckpts_{tag}', 'zip', '.', CKPT_ROOT)
    zips.append(f'grand_ckpts_{tag}.zip')
for z in zips:
    print(z, f'{os.path.getsize(z)/1e6:.0f} MB')
DRIVE = '/content/drive/MyDrive'
if os.path.isdir(DRIVE):
    dst = os.path.join(DRIVE, 'grand_sweep')
    os.makedirs(dst, exist_ok=True)
    for z in zips:
        shutil.copy(z, dst)
    print('copied to Drive:', dst)
else:
    from google.colab import files
    for z in zips:
        files.download(z)